# Example 2.2 / 5: Tuple Keys in Python

Companion notebook for **2.2 Lists, Tables and Slices in Data Files** in the AMPL lab Documentation, section "Tuple Keys in Python". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Tuple Keys in Python

Python dictionaries avoid presentation slices: each tuple key gives every index explicitly.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PLANT;                                  # Plants: fixed in each slice
set PROD;                                   # Products: rows of each slice
set PERIOD;                                 # Periods: columns of each slice
param capacity {PLANT, PROD, PERIOD} >= 0;  # Capacity by plant, product and period

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PLANT"] = ["east", "west"]
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["PERIOD"] = [1, 2]
ampl.param["capacity"] = {
    ("east", "bands", 1): 30, ("east", "bands", 2): 32,
    ("east", "coils", 1): 20, ("east", "coils", 2): 21,
    ("west", "bands", 1): 25, ("west", "bands", 2): 27,
    ("west", "coils", 1): 18, ("west", "coils", 2): 19,
}

In [5]:
# run.py
print(ampl.param["capacity"]["west", "coils", 2])

19


### Pure AMPL: data.dat and model.run

Indexing a `Parameter` object with a full key returns the stored value itself (here an `int`).

The same values can be grouped into one table per plant. In `[east,*,*]`, rows supply `PROD`
and headers supply `PERIOD`:

In [6]:
%%writefile data.dat
set PLANT := east west;
set PROD := bands coils;
set PERIOD := 1 2;
param capacity :=
  [east,*,*]: 1 2 :=
    bands 30 32
    coils 20 21
  [west,*,*]: 1 2 :=
    bands 25 27
    coils 18 19;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display capacity['west','coils',2];

capacity['west','coils',2] = 19

